In [53]:
from comet_ml import Experiment

import importlib
import sys
sys.path.append("../utils")

import images_dataset
importlib.reload(images_dataset)
from images_dataset import train_val_dataset

train_dataset, val_dataset = train_val_dataset()

Identified 80 unique labels for 4970 files.
Found 4970 files belonging to 1 classes.
Using 3976 files for training.
Using 994 files for validation.


In [54]:
from tensorflow.keras import mixed_precision
import tensorflow as tf

# Configurar Mixed Precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

In [55]:
import tensorflow as tf

normalization_layer = tf.keras.layers.Rescaling(1./255)
train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
val_dataset = val_dataset.map(lambda x, y: (normalization_layer(x), y))

In [56]:
def ConvBlock(x, filters, downsample=False):
    residual = x
    if downsample:
        residual = tf.keras.layers.Conv2D(filters, (1, 1), strides=(2, 2), padding='same', use_bias=False)(residual)
        residual = tf.keras.layers.BatchNormalization()(residual)
        x = tf.keras.layers.Conv2D(filters, (3, 3), strides=(2, 2), padding='same', use_bias=False)(x)
    else:
        if x.shape[-1] != filters:
            residual = tf.keras.layers.Conv2D(filters, (1, 1), padding='same', use_bias=False)(residual)
            residual = tf.keras.layers.BatchNormalization()(residual)
        x = tf.keras.layers.Conv2D(filters, (3, 3), padding='same', use_bias=False)(x)

    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2D(filters, (3, 3), padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.Add()([x, residual])
    x = tf.keras.layers.ReLU()(x)

    return x

def create_model(input_shape, num_classes):
    inputs = tf.keras.layers.Input(shape=input_shape)
    
    x = ConvBlock(inputs, 64)
    x = ConvBlock(x, 128, downsample=True)
    x = ConvBlock(x, 256, downsample=True)
    x = ConvBlock(x, 512, downsample=True)
    
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
    
    model = tf.keras.models.Model(inputs, outputs)
    return model

In [57]:
# Ajusta input_shape y num_classes según tus datos
num_classes = 80
input_shape = (300, 300, 3)
model = create_model(input_shape, num_classes)
model.summary()

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

num_epochs = 10  # Ajusta el número de épocas según tu preferencia
model.fit(train_dataset, epochs=num_epochs, validation_data=val_dataset)

Model: "model_7"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_17 (InputLayer)          [(None, 300, 300, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d_137 (Conv2D)            (None, 300, 300, 64  1728        ['input_17[0][0]']               
                                )                                                                 
                                                                                                  
 batch_normalization_137 (Batch  (None, 300, 300, 64  256        ['conv2d_137[0][0]']             
 Normalization)                 )                                                           

COMET WARNING: tensorflow datasets are not currently supported for gradient and activation auto-logging


Epoch 1/10
497/497 [==============================] - 92s 167ms/step - loss: 4.9028 - accuracy: 0.0460 - val_loss: 11.2806 - val_accuracy: 0.0000e+00
Epoch 2/10
497/497 [==============================] - 83s 165ms/step - loss: 4.6323 - accuracy: 0.0727 - val_loss: 5.6770 - val_accuracy: 0.0664
Epoch 3/10
497/497 [==============================] - 81s 162ms/step - loss: 4.7382 - accuracy: 0.0915 - val_loss: 4.7044 - val_accuracy: 0.1036
Epoch 4/10
497/497 [==============================] - 81s 162ms/step - loss: 4.9670 - accuracy: 0.0898 - val_loss: 6.8012 - val_accuracy: 0.0392
Epoch 5/10
497/497 [==============================] - 81s 162ms/step - loss: 5.2984 - accuracy: 0.0931 - val_loss: 45.4669 - val_accuracy: 0.0111
Epoch 6/10
497/497 [==============================] - 82s 164ms/step - loss: 5.7707 - accuracy: 0.0913 - val_loss: 32.0375 - val_accuracy: 0.0181
Epoch 7/10
497/497 [==============================] - 81s 162ms/step - loss: 6.1445 - accuracy: 0.0938 - val_loss: 8.0355 -